# Setup

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 72
from IPython.display import Markdown
import numpy as np
import os

from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
# load_dotenv('../teachingk ey.env')

In [2]:
# API configuration
Client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY', '<your OpenAI API key if not set as env var>'))

I chose ChatGPT (gpt-5.4) because it's the most recent model providing strong reasoning, adherence to structured prompts, and consistent multi-step interactions (which I need for the Pac Man game).

Temperature 0.6 is used as a default to provide a balance between deterministic and creative outputs. This is useful for the technical analysis (Q1), creative writing (Q2), and the Pac-Man game which needs consistent, but flexible, behavior (Q3).

Note: The code has been updated to use the more recent Responses API per this guide https://developers.openai.com/api/docs/guides/migrate-to-responses?update-item-definitions=responses#migrating-from-chat-completions.

In [3]:
def set_openai_params(model='gpt-5.4', temp=0.6, max_tokens=768, top_p=1, frequency_penalty=0, presence_penalty=0):
    openai_params = {
        'model': model,
        'temperature': temp,
        'max_tokens': max_tokens,
        'top_p': top_p,
        'frequency_penalty': frequency_penalty,
        'presence_penalty': presence_penalty
    }
    return openai_params

def get_response(params, prompt):
    # Responses API uses max_output_tokens, not max_tokens; penalties are chat-completions-only.
    return Client.responses.create(
        model=params['model'],
        input=prompt,
        temperature=params['temperature'],
        max_output_tokens=params['max_tokens'],
        top_p=params['top_p'],
    )


# Problem 1

Below is an example of an ineffective prompt. It is vague, does not specify an audience, and has no constraints on length, format, or focus. It should lead to a generic repsonse.

In [24]:
# a generic prompt
params = set_openai_params()
prompt = '“Tell me about risks of AI in drug discovery.”'

response = get_response(params, prompt)
Markdown(response.output_text)

AI can speed up drug discovery, but it also introduces important risks across science, safety, ethics, and operations.

## Main risks of AI in drug discovery

### 1. Bad data leads to bad models
AI models depend heavily on training data. In drug discovery, data can be:
- noisy
- incomplete
- biased toward well-studied targets
- inconsistent across labs and assays

If the data is poor, the model may make confident but wrong predictions.

### 2. Limited real-world generalization
A model may perform well on historical datasets but fail on new chemistry, new biological targets, or different patient populations. This is a major problem because drug discovery often involves exploring novel space.

### 3. False positives and wasted resources
AI may identify compounds that look promising computationally but fail in:
- wet-lab validation
- toxicity studies
- animal models
- clinical trials

This can waste time and money if teams trust model outputs too much.

### 4. Lack of interpretability
Some AI systems, especially deep learning models, can be hard to explain. Scientists may not know:
- why a compound was prioritized
- which features drove a prediction
- whether the model learned real biology or spurious correlations



The response has some useful ideas, but it's pretty generic. Phrases like "bad data can lead to bad outcomes" and the result that "AI systems can be hard to explain" are broad. There are some specifics "wet-lab validation", "toxicity studies" referred to, but with no detail or supporting examples. 

The response has a generic structure (numbered list), no clear target audience, a generic tone, and is very surface-level. 

Below is a more specific prompt that specifies areas to focus on, a target audience, a response format, limits to the three most important points and asks for examples with references.

In [ ]:
# a specific prompt
params = set_openai_params()
prompt = """Provide a structured analysis of the risks of AI-driven drug discovery. Include:

    Data-related risks (e.g., bias, data quality)
    Model-related risks (e.g., interpretability, overfitting)
    Regulatory and ethical risks

Format your response as a numbered list with brief explanations (2 sentences each), suitable for a technical audience. Provide the three most important points..

Cite examples with measurements to backup your claims. Cite sources and provide references. Validate each reference."""

response = get_response(params, prompt)
Markdown(response.output_text)

1. **Data bias and poor data quality can systematically distort model outputs.**  
   AI drug-discovery models are often trained on historical assay, omics, and clinical datasets that are incomplete, noisy, and heavily skewed toward well-studied targets; for example, analyses of ChEMBL and related bioactivity corpora have shown strong target-class imbalance, with kinase and GPCR families disproportionately represented relative to the broader proteome, which can inflate apparent performance while reducing generalization to underrepresented biology. In experimental screening, reproducibility is also a major issue: a large preclinical replication effort reported that only **6 of 53 landmark studies (11%)** could be confirmed, illustrating how low-quality upstream data can propagate directly into unreliable AI predictions and prioritization decisions [1,2].

2. **Model overfitting and limited interpretability create high risk of false confidence.**  
   Many molecular property and hit-identification models perform well on random train/test splits but degrade substantially under more realistic scaffold splits, indicating memorization of chemotypes rather than learning transferable structure–activity relationships; for instance, MoleculeNet benchmark results show notable drops in ROC-AUC/PRC when moving from random to scaffold-based evaluation across several tasks, and follow-on studies have argued that shortcut learning is common in ligand-based prediction pipelines [3,4]. This is compounded by weak interpretability: even when deep models rank compounds accurately, they often do not provide mechanistically reliable explanations, making it difficult to distinguish a biologically meaningful hypothesis from an artifact and increasing downstream experimental and financial risk [4,5].

3. **Regulatory and ethical risks arise from poor traceability, limited validation, and potential inequity.**  
   Regulators increasingly expect transparency, reproducibility, and lifecycle control for AI-enabled tools, yet many drug-discovery models are trained on proprietary or weakly documented datasets, making independent validation difficult; this is problematic in a field where the overall probability of success from **phase I to approval is only about 7.9%–10%**, so errors introduced early in AI-guided discovery can be extremely costly [6,7]. Ethical concerns also include population underrepresentation in genomic and clinical data—one widely cited review found that **78% of participants in genome-wide association studies were of European ancestry**, raising the risk that AI-derived target selection, biomarker strategies, or safety predictions may be less valid for other populations [8].

## References

1. Begley CG, Ellis LM. *Drug development: Raise standards for preclinical cancer research.* Nature. 2012;483(7391):531-533. doi:10.1038/483531a  
2. Errington TM, Denis A, Perfito N, et al. *Investigating the replicability of preclinical cancer biology.* eLife. 2021;10:e71601. doi:10.7554/eLife.71601  
3. Wu Z, Ramsundar B, Feinberg EN, et al. *MoleculeNet: a benchmark for molecular machine learning.* Chem Sci. 2018;9(2):513-530. doi:10.1039/C7SC02664A  
4. Wallach I, Heifets A. *Most ligand-based classification benchmarks reward memorization rather than generalization.* J Chem Inf Model. 2018;58(5):916-932. doi:10.1021/acs.jcim.7b00403  
5. Rudin C. *Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead.* Nat Mach Intell. 2019;1:206-215. doi:10.1038/s42256-019-004

We see the response is better structures, targeted to a technical audience, covers the key points requested with examples, and includes validated (real, not hallucinated) references. 

# Problem 2

The first prompt is generic with no target audience.

In [ ]:
# a general prompt
params = set_openai_params()
prompt = "Write about a technology where people can share dreams."

response = get_response(params, prompt)
Markdown(response.output_text)

Imagine a technology called a **Dream Network**: a system that lets people record, transmit, and experience each other’s dreams.

It would work through advanced brain-computer interfaces that monitor neural activity during sleep, especially during REM stages when vivid dreaming happens. The device would not simply make a video of the dream, because dreams are not like ordinary films. Instead, it would translate patterns of emotion, imagery, memory, sound, and sensation into a digital format. Another person could then “play back” the dream through a compatible headset or neural link, experiencing it as a reconstructed immersive environment.

This technology would change communication in powerful ways. People often struggle to explain feelings or memories with words, but dreams are deeply symbolic and emotional. Sharing a dream could let someone express fear, grief, love, or hope more directly than conversation. Artists could publish dream experiences as a new form of storytelling. Therapists might use shared dreams to better understand trauma, anxiety, or recurring nightmares. Friends, partners, and family members might feel closer by entering one another’s inner worlds.

At the same time, dream-sharing would raise serious ethical questions. Dreams are private, unstable, and sometimes reveal hidden desires or fears that the dreamer may not fully understand. There would need to

The response is somewhat rambling, it includes what may or may not be relevant information, e.g., "dream sharing would raise serious ethical concerns," and is not targeted to a specific audience.

The following prompt specifies a desired content length, puts a focus on emotional and sensory experiences, and specifies a target audience. The temp is set to 0.4 to allow additional variation between multiple runs for this creative exercise.

In [37]:
# a specific prompt
params = set_openai_params(temp=0.4)
prompt = """Write a short imaginative essay (150-200 words) describing the pleasures of a future technology that 
allows individuals to create, share, and experience custom-designed dreamscapes. Focus on emotional and sensory 
experiences, and use vivid descriptive language suitable for a high-school aged audience."""

response = get_response(params, prompt)
Markdown(response.output_text)

In the future, people may not just watch movies or play games—they may step into dreams they have designed themselves. Imagine slipping on a silver headband at night and choosing a world as easily as picking a song. Within moments, you could be walking through a glowing forest where the trees hum softly, or floating above a city made of glass and starlight.

The greatest pleasure of this technology would be how deeply personal it feels. A dreamscape could be built from your happiest memories, favorite colors, and secret wishes. If you missed someone, you might meet them beside a warm ocean with lavender waves. If you felt tired or afraid, you could rest in a meadow where the air tastes like honey and the sky pulses with gentle gold.

Sharing these dreamscapes would make them even more magical. Friends could invite one another into worlds filled with laughter, music, and impossible beauty. One person might create a mountain of clouds; another, a library where stories bloom like flowers. In such dreams, imagination would no longer be trapped inside the mind—it would become a place you could visit, feel, and share.

We see the response has vivid description and metaphor, e.g., "choosing a world as easily as picking a song," and is appropriate for a high-school audience. It uses vivid language, "a warm ocean with lavender waves", and matches the desired content length.

# Problem 3

For this problem I used ChatGPT with this prompt:

"You and I are going to play a text-based version of Pac-Man.
You will ask me for Pac-Man's next move (up, down, left, or right)
at each step, and I will respond with the direction I want to move.

Pac-Man will aim to eat all the dots on the grid while avoiding
the ghost. Eating a Power Pellet makes the ghost temporarily
vulnerable. While the ghost is vulnerable, Pac-Man can eat the ghost for bonus
points. If the ghost catches Pac-Man, he loses a life. The game
ends when all dots are eaten (player wins) or Pac-Man loses all
3 lives (player loses).

Grid layout:
Use a 5x5 grid using these symbols:
  - P = Pac-Man
  - G = Ghost (dangerous)
  - g = Ghost (vulnerable)
  - . = Regular dot
  - o = Power Pellet
  - \# = Wall
  - ' ' = Empty space

Start config:
- Pac-Man starts at the bottom left (row 4, col 0)
- Ghost starts at the top right (row 0, col 4)
- Place one Power Pellet at (row 0, col 0) and one at (row 4, col 4)
- Add four wall (#) tiles to create simple corridors
- Fill the remaining open cells with regular dots

Game Rules:
1. Each turn, Pac-Man moves one cell in the chosen direction
2. Pac-Man cannot move through walls or outside the grid
3. Eating a dot removes it and adds 10 points to the score
4. Eating a Power Pellet adds 50 points and makes the ghost
   vulnerable for 5 turns. While vulnerable, the ghost is shown as lowercase g
5. While the ghost is vulnerable, if Pac-Man moves onto it, Pac-Man
   eats it for 200 points and the ghost respawns at its start position
6. While the ghost is vulnerable, it should attempt to move one step away from Pac-Man each turn
7. While the ghost is not vulnerable, if the ghost and Pac-Man
   meet on the same cell, Pac-Man loses a life and resets to his start position
8. While the ghost is not vulnerable, it should move one step closer to Pac-Man each turn

Your Tasks:
1. Print the 5x5 grid at the start and after every move
2. Show the current score, lives remaining, whether the Power
   Pellet effect is active, and how many turns remain
3. Ask me: "Which direction should Pac-Man move? (up/down/left/right)"
4. After each move, describe what happened (dot eaten, ghost avoided,
   life lost, etc.)
5. Announce when the game is won or lost

Important: Keep track of the dots. Each time Pac-Man eats a dot, it must disappear and should not be rendered again. There should be no dot on Pac-Man's starting cell. The dots should not reset until the game is over even if Pac-Man loses a life or captures the ghost.

Begin by printing the initial grid and asking for my first move."

I used this Wikipedia article to review the rules for Pac-Man: https://en.wikipedia.org/wiki/Pac-Man 

The note at the end of the instrurctions about keeping track of the dots was added after I tested the game and saw that the dots were sometimes being added back.

## These first steps show the basic functionality
![Diagram](pac-man-pics/p1.png)
![Diagram](pac-man-pics/p2.png)
![Diagram](pac-man-pics/p3.png)
![Diagram](pac-man-pics/p4.png)
![Diagram](pac-man-pics/p5.png)
![Diagram](pac-man-pics/p6.png)

## Pac-Man capturing a power pellet
![Diagram](pac-man-pics/p7.png)
![Diagram](pac-man-pics/p8.png)
## The ghost moves away
![Diagram](pac-man-pics/p9.png)
![Diagram](pac-man-pics/p10.png)
## Pac-Man gets caught by the ghost
![Diagram](pac-man-pics/p11.png)
![Diagram](pac-man-pics/p12.png)
## This skips to the end of the game where Pac-Man has 1-life left and runs into the ghost
![Diagram](pac-man-pics/p13.png)
![Diagram](pac-man-pics/p14.png)